In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder\
        .appName("scenari_5")\
        .master("local[*]")\
        .config("spark.executor.memory","6g")\
        .config("spark.driver.memory","4g")\
        .config("spark.sql.shuffle.partitions","50")\
        .config("spark.sql.adaptive.enabled","true")\
        .config("spark.sql.autoBroadcastJoinThreshold","-1")\
        .config("spark.python.worker.reuse","False")\
        .config("spark.executor.heartbeatInterval","60s")\
        .config("spark.network.timeout","300s")\
        .config("spark.hadoop.io.native.lib.available","False")\
        .config("spark.serializer","org.apache.spark.serializer.KryoSerializer")\
        .getOrCreate()


In [2]:
event_dir="E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/*/"
df1=spark.read.option("mergeSchema", True).parquet(event_dir)
df1.show()

+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+-------------+------------+
|   event_id| user_id|session_id|event_type|    event_timestamp|         page_name|duration_seconds|device_type|     os|country|app_version|ab_test_group|feature_flag|
+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+-------------+------------+
|E0000666708|U0000803| S00269219|     click|2024-01-01 00:00:08|           profile|            NULL|     mobile|    Mac|     US|        3.5|         NULL|        NULL|
|E0000429699|U0020459| S00498965|    logout|2024-01-01 00:00:11|order_confirmation|            NULL|     mobile|    iOS|     US|        4.0|         NULL|        NULL|
|E0000209181|U0007420| S00189644| page_view|2024-01-01 00:00:16|          checkout|           252.0|    desktop|Android| Brazil|        4.1|         NULL|      

In [3]:
df2=df1.groupBy("user_id")\
    .agg(concat(round((sum('duration_seconds')/60),2))\
    .alias("total_duration(minutes)"))

print(df2.rdd.getNumPartitions())

df2=df2.repartition(8)

df2.write\
    .mode("overwrite")\
    .option("header",True)\
    .parquet("E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/output_repartiton/")


16


In [4]:
df2=df1.groupBy("user_id")\
    .agg(concat(round((sum('duration_seconds')/60),2))\
    .alias("total_duration(minutes)"))


df2=df2.coalesce(8)

df2.write\
    .mode("overwrite")\
    .option("header",True)\
    .parquet("E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/output_coalesec/")
